In [1]:
import sys 
from pathlib import Path
sys.path.append(str(Path(__name__).resolve().parents[2]))
import importlib
import utils 
importlib.reload(utils)
from utils.dataset import ProcessedDataset, load_geneset_dataset, split_by_indices, save_predicitons
from utils.dna import Genes, create_template_from_genes
from pathlib import Path

import numpy as np 
import pandas as pd 
from tqdm import tqdm 
import pickle 
import matplotlib.pyplot as plt

In [2]:
import utils.dataset 
import models.mlp
importlib.reload(utils.dataset)
importlib.reload(models.mlp)
from utils.dataset import get_raw_dataset
# train test split for combined features
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA 

DATA_DIR = Path(__name__).resolve().parents[2] / 'data'
ROOT_DIR = Path(__name__).resolve().parents[2]

GENESET = "test"
DATASET_NAME = 'test_geneset_embeddings'
FEATURES = "arcface-r100"
DATASET_DIR = DATA_DIR / DATASET_NAME
EXPERIMENT_DIR = ROOT_DIR / 'experiments' / DATASET_NAME / "diffusion" / FEATURES
TEST_DATASET_DIR = DATA_DIR / 'prison_sample_embeddings'

geneset = ProcessedDataset(DATASET_DIR)
images = geneset.alligned_images
X, Y = get_raw_dataset(GENESET, FEATURES)
X_train, X_test, Y_train, Y_test, images_train, images_test = train_test_split(X, Y, images, test_size=0.1, random_state=42)
scaler_x = MinMaxScaler()
scaler_x.fit(X_train)
X_train = scaler_x.transform(X_train)
X_test = scaler_x.transform(X_test)
print(f"shape of dataset: {X.shape}, {Y.shape}")

real_dataset = ProcessedDataset(TEST_DATASET_DIR)
real_images = real_dataset.alligned_images
Y_real = real_dataset.model2normalized_embeddings[FEATURES]
print(f"shape of real dataset: {Y_real.shape}")

/Users/czyjtu/dev/prompt2avatar/venv/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 2/2 [00:00<00:00, 93.21it/s]


shape of dataset: (14990, 3), (14990, 512)


100%|██████████| 9/9 [00:00<00:00, 936.32it/s]

shape of real dataset: (1000, 512)


# Model

In [3]:
import importlib
import models.diffusion.ddpm as ddpm
import models.callbacks as callbacks
importlib.reload(ddpm)
importlib.reload(callbacks)
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import WandbLogger
import torch 
from models.callbacks import MetricsCallback
from models.diffusion.ddpm import ConditionalDDPM, SimpleCondNet, ConditionalNetworkConfig
import torch.utils.data 
import wandb 

TRAIN = True
MODEL_CHECKPOINT = "TBD"
WANDB_DIR = Path("/Users/czyjtu/dev/prompt2avatar/checkpoints/diffusion") / GENESET
WANDB_DIR.mkdir(exist_ok=True, parents=True)


X_th = torch.tensor(X_train).float()
Y_th = torch.tensor(Y_train).float()

train_dataset = torch.utils.data.TensorDataset(X_th, Y_th)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)

test_dataset = torch.utils.data.TensorDataset(torch.tensor(X_test).float(), torch.tensor(Y_test).float())
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)

if TRAIN:
    with wandb.init(
        project="prompt2avatar", entity="czyjtu",
        tags=["diffusion", GENESET], group="diffusion" + GENESET, dir=WANDB_DIR
        ) as wandb_run:
        checkpoint_callback = ModelCheckpoint(
            monitor='val_loss',
            dirpath=EXPERIMENT_DIR / 'model_checkpoints',
            filename='best_model',
            save_top_k=1,
            mode='min'
        )
        wandb_logger = WandbLogger(project='prompt2avatar', entity="czyjtu", log_model="all", experiment=wandb_run)
        net_config = ConditionalNetworkConfig(
            input_dim=X_th.shape[1], 
            condition_dim=Y_th.shape[1], 
            condition_emb_dim=64, 
            inner_dim=[64, 64, 64], 
            condition_encoder_dim=[64, 64, 64]
        )
        cond_ddpm = ConditionalDDPM(net_config)
        # print(Y_ltr.shape())
        callback = MetricsCallback(
            torch.tensor(images_test).float(),
            torch.tensor(X_test).float(), 
            torch.tensor(Y_test).float(), 
            scaler_x
        )

        trainer = pl.Trainer(
            max_epochs=1000, 
            callbacks=[callback, checkpoint_callback], 
            logger=wandb_logger, 
            accelerator="mps"
            )
        trainer.fit(cond_ddpm, train_dataloaders=train_loader, val_dataloaders=test_loader)
        cond_ddpm.chec
else:
    cond_ddpm = ConditionalDDPM.load_from_checkpoint(MODEL_CHECKPOINT)
    cond_ddpm = cond_ddpm.to("cpu")
    cond_ddpm.eval()

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: czyjtu. Use `wandb login --relogin` to force relogin


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/Users/czyjtu/dev/prompt2avatar/venv/lib/python3.10/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:653: Checkpoint directory /Users/czyjtu/dev/prompt2avatar/experiments/test_geneset_embeddings/diffusion/arcface-r100/model_checkpoints exists and is not empty.

  | Name | Type          | Params
---------------------------------------
0 | net  | SimpleCondNet | 79.3 K
---------------------------------------
79.3 K    Trainable params
0         Non-trainable params
79.3 K    Total params
0.317     Total estimated model params size (MB)


Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

/Users/czyjtu/dev/prompt2avatar/venv/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:441: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


/Users/czyjtu/dev/prompt2avatar/venv/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 250:  68%|██████▊   | 72/106 [00:02<00:01, 30.79it/s, v_num=w6oc, training_loss=0.0421, val_loss=0.0228, RMSE=0.030, Scaled RMSE=4.290, MAE=0.0232, R2=0.954]   

In [ ]:
cond_ddpm.exal().to("cpu")
Y_test_th = torch.tensor(Y_test).float()
X_test_pred = cond_ddpm.sample(len(Y_test), show_progress=False, Y=Y_test_th, return_intemediates=False)[-1].detach().numpy()
save_predicitons(X_test_pred, scaler_x, EXPERIMENT_DIR / 'predicted_dnas_diffusion', GENESET)

In [15]:
cond_ddpm.exal().to("cpu")
Y_real_th = torch.tensor(Y_real).float()
X_real_pred = cond_ddpm.sample(len(Y_real), show_progress=False, Y=Y_real_th, return_intemediates=False)[-1].detach().numpy()
save_predicitons(X_real_pred, scaler_x, EXPERIMENT_DIR / 'predicted_real_dnas_diffusion', GENESET)

100%|██████████| 1/1 [00:00<00:00, 89.57it/s]


In [17]:
1500 * 4 / 60 / 60


1.6666666666666667

In [18]:
len(X_test_sc)

1499

In [20]:
len(X_train)

13491